In [8]:
# We install the specific Unsloth branch that supports Gemma 3n Vision
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

print("Unsloth and dependencies installed! Ready for Gemma 3n.")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-5t5i318x/unsloth_10badeaadb914235abbe833ad8ab78a7
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-5t5i318x/unsloth_10badeaadb914235abbe833ad8ab78a7
  Resolved https://github.com/unslothai/unsloth.git to commit ded942c765abf22fc8e3b0a67f847b83a6b2ad53
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Unsloth and dependencies installed! Ready for Gemma 3n.


In [9]:
import kagglehub
from huggingface_hub import notebook_login
import os

# 1. Login to Hugging Face
# You will be prompted to enter your token (Get it from https://huggingface.co/settings/tokens)
print("Please login to Hugging Face to access Gemma 3n:")
notebook_login()

# 2. Download Dataset
# We use a small animal dataset as a placeholder for the prototype structure.
# In a real run, you would use: path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("\nDownloading dataset...")
path = kagglehub.dataset_download("ashishsaxena2209/animal-image-datasetdog-cat-and-panda")

base_dir = path
print(f"Dataset ready at: {base_dir}")

Please login to Hugging Face to access Gemma 3n:


100%|██████████| 376M/376M [00:07<00:00, 52.4MB/s]

Extracting files...


Dataset ready at: /root/.cache/kagglehub/datasets/ashishsaxena2209/animal-image-datasetdog-cat-and-panda/versions/1


In [10]:
from trl import SFTTrainer, SFTConfig
from unsloth import FastVisionModel

# 1. Configure Training
training_args = SFTConfig(
    output_dir = "gemma-3n-plant-village",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60, # Increase to 200-500 for better results in a real run
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    dataset_num_proc = 2,
    packing = False,
    remove_unused_columns = False, # Critical for multimodal datasets
)

# 2. Initialize Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    data_collator = FastVisionModel.get_data_collator(model, tokenizer),
    args = training_args,
)

# 3. Train
print("Starting Gemma 3n training...")
trainer.train()
print("Training Complete!")

NameError: name 'ds' is not defined

In [11]:
from trl import SFTTrainer, SFTConfig
from unsloth import FastVisionModel
from datasets import Dataset, Image
import glob
import kagglehub
import torch

# --- PART 1: ENSURE DATASET EXISTS (Fix for NameError) ---
print("Checking dataset...")

# 1. Re-verify data path (uses cache, so it's fast)
# Note: Using the animal dataset as placeholder for the prototype structure
path = kagglehub.dataset_download("ashishsaxena2209/animal-image-datasetdog-cat-and-panda")
base_dir = path

# 2. Re-create dataset if 'ds' is missing
# We rebuild the dataset here to ensure 'ds' is always defined
image_files = glob.glob(f"{base_dir}/**/*.jpg", recursive=True)[:200] # Limit to 200 for prototype

dataset_data = []
instruction = "You are an expert plant pathologist. Analyze this leaf image and identify the disease."

for file_path in image_files:
    label = file_path.split('/')[-2]
    dataset_data.append({
        "image": file_path,
        "messages": [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]},
            {"role": "assistant", "content": [{"type": "text", "text": label}]}
        ]
    })

ds = Dataset.from_list(dataset_data)
ds = ds.cast_column("image", Image())
print(f"Dataset ready with {len(ds)} samples.")

# --- PART 2: TRAINING ---

# 1. Configure Training
training_args = SFTConfig(
    output_dir = "gemma-3n-plant-village",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    dataset_num_proc = 2,
    packing = False,
    remove_unused_columns = False,
    dataset_kwargs = {"skip_prepare_dataset": True}, # We already prepared it above
)

# 2. Initialize Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    data_collator = FastVisionModel.get_data_collator(model, tokenizer),
    args = training_args,
)

# 3. Train
print("Starting Gemma 3n training...")
trainer.train()
print("Training Complete!")

Checking dataset...
Using Colab cache for faster access to the 'animal-image-datasetdog-cat-and-panda' dataset.
Dataset ready with 200 samples.


AttributeError: type object 'FastVisionModel' has no attribute 'get_data_collator'